## **Execução de RPC + Ollama no Google Colab**  

Este tutorial apresenta um guia detalhado para a configuração e execução de **RPC + Ollama** em um ambiente **Google Colab**.  

### **Requisitos**  

Antes de iniciar, certifique-se de atender aos seguintes requisitos:  

- Conta no **Ngrok** com um **token de autenticação** válido.  
- Acesso a um ambiente **Google Colab** com suporte a **GPU Nvidia (T4 - free tier)**.  
- **Python** instalado na máquina local.  

Os próximos passos detalham o processo de instalação e configuração, garantindo a correta execução do ambiente para testes e experimentação.



---



**I. Instalação das Dependências do Ollama**


> O pacote `pciutils` é necessário para que o Ollama consiga identificar corretamente o tipo de GPU disponível na instância do Colab.

> A instalação do Ollama na instância em tempo de execução será realizada pelo seguinte comando `sh curl -fsSL https://ollama.com/install.sh | sh`

In [ ]:
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

**II. Instalação das Dependências do Python**  

> O pacote `langchain-ollama` é necessário para integrar o Ollama com a biblioteca **Langchain**, facilitando a interação com modelos de linguagem.  

> O pacote `pyngrok` é necessário para configurar e gerenciar o túnel do Ngrok diretamente a partir do código Python.  

In [ ]:
!pip install langchain-ollama
!pip install langchain_community
!pip install pyngrok
!pip install flask

**III. Autenticar Ngrok com Authtoken**
> Esta parte é essencial para comunição fora do Colab, para obter o token basta [acessar a página](https://dashboard.ngrok.com/get-started/your-authtoken), e copiar o token após se autenticar.


In [ ]:
!ngrok authtoken <authtoken>

**IV. Execução do Ollama**  

> Para utilizar o Ollama, é necessário que ele seja executado como um serviço em segundo plano, paralelo aos seus scripts. No entanto, como os Jupyter Notebooks são projetados para rodar os blocos de código de forma sequencial, isso dificulta a execução simultânea de dois blocos de código. Como solução, vamos criar um serviço utilizando o módulo `subprocess` em Python, garantindo que a execução de uma célula não bloqueie a execução das demais.

> Além disso, para garantir que o serviço Ollama esteja completamente em funcionamento antes de baixar o modelo, utilizamos um pequeno delay com o comando `time.sleep(5)`, o que dá tempo para o serviço ser inicializado corretamente.

In [ ]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

**V. Baixando o Modelo**  

> Para utilizar o modelo LLM, é necessário fazer o download utilizando o comando `ollama pull llama3.1`. Este comando irá baixar o modelo Llama 3.1 8b, que pode ser utilizado em seu ambiente Colab.  

> Se desejar utilizar outros modelos, você pode consultar a lista completa de modelos disponíveis no site oficial do Ollama: [https://ollama.com/library](https://ollama.com/library).  

In [ ]:
!ollama pull llama3.1

**VI. Iniciar servidor RPC com Ollama e RPC**
> Este código configura um servidor XML-RPC que permite interagir com o modelo de texto rodando no Ollama + Langchain.



In [ ]:
from xmlrpc.server import SimpleXMLRPCServer
from xmlrpc.server import SimpleXMLRPCRequestHandler
from langchain import LLMChain, PromptTemplate
from langchain.llms import Ollama
from pyngrok import ngrok

import json

# Carrega o contexto para o RAG por um arquivo JSON
with open("django_models_context.json", "r") as file:
    schema_info = json.load(file)
    print(schema_info)

# Inicia o modelo LLama 3.1
llm = Ollama(model="llama3.1")

# Define o template (personalidade) a ser utilizado pelo modelo.
prompt_template = PromptTemplate(
    input_variables=["input_text", "schema_info"],
    template="""
    Given the following database schema:
    {schema_info}

    Convert the following natural language query into a SQL SELECT statement:
    {input_text}

    Remember to follow these rules riggidly:
    1. If you don't recognize one of the tables or collums on the schema. Respond with "I don't have this information."
    2. Ensure to return only the SQL query, nothing else and no formatting, text only.
    3. Ensure the query is safe and only retrieves data (SELECT only).
    """
)

# Cria a chain usando o modelo e o template
sql_chain = LLMChain(llm=llm, prompt=prompt_template)



# Função que gera uma consulta SQL
def gerar_resposta(prompt_usuario):
  print('SERVER : Nova requisição')
  print('> ',prompt_usuario)

  # Generate SQL query
  sql_query = sql_chain.run(input_text=prompt_usuario, schema_info=schema_info)

  # Validate and execute the query
  #validated_query = validate_sql(sql_query)

  #return validated_query
  return sql_query

# Cria o servidor XML-RPC
class ManipuladorDeRequisicoes(SimpleXMLRPCRequestHandler):
    rpc_paths = ("/RPC2",)

server = SimpleXMLRPCServer(("0.0.0.0", 1339), requestHandler=ManipuladorDeRequisicoes, allow_none=True)
server.register_function(gerar_resposta, "gerar_resposta")

# Inicia o túnel ngrok
url_publica = ngrok.connect(1339).public_url
print("URL Pública: ", url_publica)

print("Servidor XML-RPC em execução...")
server.serve_forever()
